# Student Final Marks Prediction
EDA, model comparison and prediction on `data/Final_Marks_Data.csv`.

Run the cells from top to bottom (Run All). Open this notebook from the project folder.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config import FEATURES, TARGET
from preprocessing import load_data, split_data, build_preprocessor
sns.set_theme(style='whitegrid')

## 1. Load and inspect the data

In [ ]:
df = load_data()
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().round(2)

In [ ]:
print('Missing values:', df.isna().sum().sum())
print('Duplicates:', df.duplicated().sum())

## 2. Exploratory data analysis

In [ ]:
sns.histplot(df[TARGET], kde=True)
plt.title('Distribution of final marks')
plt.show()

In [ ]:
sns.heatmap(df[FEATURES + [TARGET]].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation heatmap')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), FEATURES):
    sns.regplot(x=df[c], y=df[TARGET], ax=ax, scatter_kws=dict(alpha=0.25, s=10), line_kws=dict(color='red'))
    ax.set_title(f'{c} vs final marks')
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

**Observation:** every feature is positively correlated with final marks; attendance and the two internal tests have the strongest relationship.

## 3. Train and compare models

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error

X_train, X_test, y_train, y_test = split_data(df)
models = {'Linear Regression': LinearRegression(), 'Ridge': Ridge(),
          'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42),
          'Gradient Boosting': GradientBoostingRegressor(random_state=42)}
rows = []
for name, m in models.items():
    pipe = Pipeline([('prep', build_preprocessor()), ('model', m)])
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2').mean()
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    rows.append(dict(Model=name, R2=r2_score(y_test, p), MAE=mean_absolute_error(y_test, p), CV_R2=cv))
pd.DataFrame(rows).sort_values('R2', ascending=False).round(3)

## 4. Predict for a new student

In [ ]:
import joblib
model = joblib.load('../models/best_model.pkl')  # created by: python src/train_models.py
student = pd.DataFrame([[90, 30, 35, 8, 3]], columns=FEATURES)
print('Predicted final marks:', round(float(model.predict(student)[0]), 1))